# Cardiovascular Disease Prediction
## ML Deployment Project

**Dataset:** [Cardiovascular Disease dataset](https://www.kaggle.com/datasets/sulianova/cardiovascular-disease-dataset) (70,000 records)

---

## 1. Problem Definition

### ปัญหาคืออะไร?
โรคหัวใจและหลอดเลือด (Cardiovascular Disease - CVD) เป็น **สาเหตุการเสียชีวิตอันดับ 1 ของโลก** คร่าชีวิตผู้คนกว่า 17.9 ล้านคนต่อปี (WHO, 2021) ซึ่งคิดเป็นประมาณ 32% ของการเสียชีวิตทั้งหมดทั่วโลก

### ทำไมถึงน่าแก้ไขด้วย ML?
- **Early detection saves lives:** หากสามารถคัดกรองผู้ที่มีความเสี่ยงสูงได้เร็ว จะช่วยให้แพทย์วางแผนป้องกันและรักษาได้ทันท่วงที
- **ปัจจัยเสี่ยงมีหลายตัว:** ความดันโลหิต, คอเลสเตอรอล, น้ำหนัก, อายุ ฯลฯ ซึ่ง ML สามารถวิเคราะห์ความสัมพันธ์ที่ซับซ้อนเหล่านี้ได้ดีกว่าการใช้ threshold ตัวเดียว
- **Scalable screening:** โมเดลที่ deploy เป็น web app สามารถใช้เป็นเครื่องมือคัดกรองเบื้องต้นได้ โดยไม่จำเป็นต้องใช้อุปกรณ์ทางการแพทย์ที่ซับซ้อน

### Dataset
- **แหล่งที่มา:** Kaggle - Cardiovascular Disease dataset โดย Svetlana Ulianova
- **ขนาด:** 70,000 records จากข้อมูลการตรวจสุขภาพจริง
- **Features (12 ตัว):**

| Feature | คำอธิบาย | ประเภท |
|---------|----------|--------|
| age | อายุ (เป็นวัน) | Numerical |
| gender | เพศ (1=หญิง, 2=ชาย) | Categorical |
| height | ส่วนสูง (cm) | Numerical |
| weight | น้ำหนัก (kg) | Numerical |
| ap_hi | ความดันโลหิตตัวบน (Systolic) | Numerical |
| ap_lo | ความดันโลหิตตัวล่าง (Diastolic) | Numerical |
| cholesterol | ระดับคอเลสเตอรอล (1=ปกติ, 2=สูงกว่าปกติ, 3=สูงมาก) | Categorical (Ordinal) |
| gluc | ระดับน้ำตาลในเลือด (1=ปกติ, 2=สูงกว่าปกติ, 3=สูงมาก) | Categorical (Ordinal) |
| smoke | สูบบุหรี่หรือไม่ (0/1) | Binary |
| alco | ดื่มแอลกอฮอล์หรือไม่ (0/1) | Binary |
| active | ออกกำลังกายหรือไม่ (0/1) | Binary |
| **cardio** | **มีโรคหัวใจหรือไม่ (0/1) — Target** | **Binary** |

### ทำไม ML เหมาะกับปัญหานี้?
- เป็น **Binary Classification** ที่ชัดเจน (มี/ไม่มี โรคหัวใจ)
- มี features หลากหลายทั้ง numerical และ categorical ที่มีความสัมพันธ์กับ target
- Dataset มีขนาดใหญ่พอ (70,000 records) ที่จะ train โมเดลได้อย่างน่าเชื่อถือ
- ผลลัพธ์สามารถนำไปใช้ในเชิงปฏิบัติได้จริง เป็นเครื่องมือคัดกรองเบื้องต้น

## 2. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")

In [ ]:
df = pd.read_csv('cardio_train.csv', sep=';')

# ลบ column 'id' ที่ไม่จำเป็น
df = df.drop('id', axis=1)

# แปลง age จากวันเป็นปี (อ่านง่ายกว่า)
df['age'] = (df['age'] / 365.25).round(1)

print(f"Dataset shape: {df.shape}")
print(f"จำนวน records: {df.shape[0]:,}")
print(f"จำนวน features: {df.shape[1] - 1} + 1 target")
print()
df.head(10)

In [ ]:
df.info()
print()
df.describe()

In [ ]:
# ดู unique values ของ categorical features
cat_cols = ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']
for col in cat_cols:
    print(f"{col}: {sorted(df[col].unique())}")
    
print()

# ดู class balance ของ target
print("Target Distribution:")
print(df['cardio'].value_counts())
print()
print(f"Class ratio: {df['cardio'].value_counts(normalize=True).round(4).to_dict()}")
print("=> Dataset มี class balance ที่สมมาตรมาก ไม่ต้องกังวลเรื่อง imbalanced data")

## 3. Exploratory Data Analysis (EDA)

### 3.1 Missing Values & Outliers

Dataset นี้ไม่มี null values แต่มี **outliers ที่ผิดปกติอย่างชัดเจน** โดยเฉพาะในคอลัมน์ความดันโลหิต (ap_hi, ap_lo) ที่มีค่าติดลบและค่าสูงผิดปกติ (เช่น systolic = 16,020 mmHg) ซึ่งเป็นไปไม่ได้ในทางการแพทย์

In [ ]:
# ตรวจสอบ null values
print("Null values per column:")
print(df.isnull().sum())
print(f"\nTotal nulls: {df.isnull().sum().sum()}")
print("=> ไม่มี missing values เลย ไม่ต้อง impute")

print("\n" + "="*60)
print("ตรวจสอบค่าผิดปกติ (Outliers)")
print("="*60)

# ความดันโลหิต
print(f"\nap_hi (Systolic BP): min={df['ap_hi'].min()}, max={df['ap_hi'].max()}")
print(f"  - ค่าติดลบ: {(df['ap_hi'] < 0).sum()} records")
print(f"  - ค่า > 300 mmHg: {(df['ap_hi'] > 300).sum()} records")
print(f"  (ค่าปกติ: 90-180 mmHg)")

print(f"\nap_lo (Diastolic BP): min={df['ap_lo'].min()}, max={df['ap_lo'].max()}")
print(f"  - ค่าติดลบ: {(df['ap_lo'] < 0).sum()} records")
print(f"  - ค่า > 200 mmHg: {(df['ap_lo'] > 200).sum()} records")
print(f"  (ค่าปกติ: 60-120 mmHg)")

print(f"\nap_hi < ap_lo (ผิดหลักการแพทย์): {(df['ap_hi'] < df['ap_lo']).sum()} records")

# ส่วนสูงและน้ำหนัก
print(f"\nheight: min={df['height'].min()} cm, max={df['height'].max()} cm")
print(f"  - < 100 cm: {(df['height'] < 100).sum()} records")

print(f"\nweight: min={df['weight'].min()} kg, max={df['weight'].max()} kg")
print(f"  - < 30 kg: {(df['weight'] < 30).sum()} records")

### 3.2 Target Distribution (Class Balance)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
counts = df['cardio'].value_counts()
colors = ['#2ecc71', '#e74c3c']
labels = ['No CVD (0)', 'Has CVD (1)']

ax[0].bar(labels, counts.values, color=colors, edgecolor='black', alpha=0.8)
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold', fontsize=12)
ax[0].set_title('Target Distribution (Count)', fontsize=14, fontweight='bold')
ax[0].set_ylabel('Count')

# Pie chart
ax[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
          startangle=90, textprops={'fontsize': 12}, explode=(0.02, 0.02))
ax[1].set_title('Target Distribution (%)', fontsize=14, fontweight='bold')

plt.suptitle('Class Balance Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("=> Dataset มี class balance ที่เกือบสมมาตรพอดี (50/50)")
print("=> สามารถใช้ Accuracy เป็น metric หลักได้ แต่ยังควรดู Precision/Recall ด้วย")
print("   เพราะในบริบทการแพทย์ False Negative (ป่วยแต่ทำนายว่าไม่ป่วย) อันตรายกว่า")

### 3.3 Distribution ของ Numerical Features

ดู distribution ของ features ตัวเลขเพื่อเข้าใจ shape ของข้อมูล และระบุ outliers ที่ต้องจัดการ

In [ ]:
num_cols = ['age', 'height', 'weight', 'ap_hi', 'ap_lo']

fig, axes = plt.subplots(2, 5, figsize=(22, 9))

for i, col in enumerate(num_cols):
    # Histogram แยกตาม target
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        axes[0, i].hist(df[df['cardio']==label][col], bins=50, alpha=0.6, 
                        color=color, label=f'{"No CVD" if label==0 else "Has CVD"}')
    axes[0, i].set_title(f'{col}', fontsize=13, fontweight='bold')
    axes[0, i].legend(fontsize=9)
    
    # Box plot
    df.boxplot(column=col, by='cardio', ax=axes[1, i])
    axes[1, i].set_title(f'{col} by Target', fontsize=11)
    axes[1, i].set_xlabel('cardio')

plt.suptitle('Numerical Features: Distribution & Box Plots', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.4 Distribution ของ Categorical Features

ดูว่า categorical features แต่ละตัวมีความสัมพันธ์กับ target อย่างไร

In [ ]:
cat_cols = ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']
cat_labels = {
    'gender': {1: 'Female', 2: 'Male'},
    'cholesterol': {1: 'Normal', 2: 'Above Normal', 3: 'Well Above'},
    'gluc': {1: 'Normal', 2: 'Above Normal', 3: 'Well Above'},
    'smoke': {0: 'No', 1: 'Yes'},
    'alco': {0: 'No', 1: 'Yes'},
    'active': {0: 'No', 1: 'Yes'}
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    # สร้าง cross-tab แล้ว plot stacked bar
    ct = pd.crosstab(df[col], df['cardio'], normalize='index')
    ct.index = [cat_labels[col].get(idx, idx) for idx in ct.index]
    ct.columns = ['No CVD', 'Has CVD']
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['#2ecc71', '#e74c3c'], 
            alpha=0.8, edgecolor='black')
    axes[i].set_title(f'{col}', fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Proportion')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=0)
    axes[i].legend(fontsize=9)
    axes[i].axhline(y=0.5, color='black', linestyle='--', alpha=0.3)

plt.suptitle('Categorical Features vs Target (Proportion)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3.5 Correlation Matrix

In [ ]:
plt.figure(figsize=(12, 9))
corr = df.corr(numeric_only=True)

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, vmin=-1, vmax=1,
            cbar_kws={'shrink': 0.8})

plt.title('Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# แสดง features ที่ correlate กับ target มากที่สุด
target_corr = corr['cardio'].drop('cardio').abs().sort_values(ascending=False)
print("Features ที่ correlate กับ target (cardio) มากที่สุด:")
for feat, val in target_corr.items():
    bar = '█' * int(val * 50)
    print(f"  {feat:15s} {val:.3f} {bar}")

### 3.6 สรุป EDA

**สิ่งที่ค้นพบ:**
1. **Outliers รุนแรง**: ap_hi และ ap_lo มีค่าที่เป็นไปไม่ได้ทางการแพทย์ (ค่าติดลบ, ค่าหลักหมื่น) → ต้องลบออก
2. **ap_hi < ap_lo**: มี 1,234 records ที่ systolic < diastolic ซึ่งผิดปกติ → ต้องลบออก  
3. **Cholesterol และ Age** เป็น features ที่ correlate กับ target มากที่สุด
4. **Class Balance ดีมาก** (50/50) → ไม่ต้องทำ oversampling/undersampling
5. **ap_hi/ap_lo มี correlation ต่ำ** เพราะ outliers บิดเบือน → หลัง clean จะดีขึ้น
6. **Smoke, Alco, Gender** มี correlation ต่ำมากกับ target แต่ยังคงเก็บไว้เพราะอาจมีผลร่วมกับ features อื่น

## 4. Data Preprocessing & Pipeline

### แผนการ:
1. **ลบ outliers** ที่เป็นไปไม่ได้ทางการแพทย์ (ไม่ใช้ IQR เพราะจะ aggressive เกินไป แต่ใช้ domain knowledge กำหนดขอบเขตที่สมเหตุสมผล)
2. **สร้าง BMI feature** จาก weight และ height เพราะ BMI เป็นตัวชี้วัดสุขภาพที่สำคัญ
3. **แยก features** เป็น numerical กับ categorical
4. **สร้าง Pipeline** ที่รวม preprocessing (StandardScaler) เข้ากับโมเดล เพื่อป้องกัน data leakage

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (classification_report, confusion_matrix, 
                              roc_auc_score, roc_curve, f1_score,
                              precision_score, recall_score, accuracy_score)

print("Sklearn modules imported successfully!")

In [ ]:
print(f"ก่อนทำความสะอาด: {df.shape[0]:,} records")

# 1. ลบ outliers ด้วย domain knowledge (ไม่ใช่ IQR)
# ค่าความดันโลหิตที่สมเหตุสมผลทางการแพทย์
df_clean = df[
    (df['ap_hi'] >= 80) & (df['ap_hi'] <= 250) &   # Systolic: 80-250 mmHg
    (df['ap_lo'] >= 40) & (df['ap_lo'] <= 160) &   # Diastolic: 40-160 mmHg
    (df['ap_hi'] > df['ap_lo']) &                    # Systolic ต้อง > Diastolic
    (df['height'] >= 120) & (df['height'] <= 220) & # ส่วนสูง: 120-220 cm
    (df['weight'] >= 30) & (df['weight'] <= 200)    # น้ำหนัก: 30-200 kg
].copy()

removed = df.shape[0] - df_clean.shape[0]
print(f"หลังทำความสะอาด: {df_clean.shape[0]:,} records")
print(f"ลบออก: {removed:,} records ({removed/df.shape[0]*100:.1f}%)")
print()

# 2. สร้าง BMI feature
df_clean['bmi'] = df_clean['weight'] / (df_clean['height'] / 100) ** 2
df_clean['bmi'] = df_clean['bmi'].round(1)

print(f"BMI statistics:")
print(f"  Mean: {df_clean['bmi'].mean():.1f}")
print(f"  Min: {df_clean['bmi'].min():.1f}, Max: {df_clean['bmi'].max():.1f}")
print()

# ตรวจสอบว่า class balance ยังดีอยู่
print("Class balance หลัง clean:")
print(df_clean['cardio'].value_counts(normalize=True).round(4))

### 4.1 Feature Selection

ไม่ใช่ทุก feature จะช่วยทำนาย — ต้องวิเคราะห์ก่อนว่าตัวไหนมีความสัมพันธ์กับ target จริง

เกณฑ์ในการคัดเลือก:
- **Correlation กับ cardio** ≥ 0.03 (มีความสัมพันธ์ที่พอวัดได้)
- **Domain knowledge** — feature ต้องมีเหตุผลทางการแพทย์ที่สัมพันธ์กับโรคหัวใจ

In [ ]:
# วิเคราะห์ correlation ของแต่ละ feature กับ cardio
all_features = ['age', 'height', 'weight', 'ap_hi', 'ap_lo', 'bmi',
                'gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']

corr_with_target = df_clean[all_features + ['cardio']].corr()['cardio'].drop('cardio').abs().sort_values(ascending=False)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#2ecc71' if v >= 0.03 else '#e74c3c' for v in corr_with_target.values]
corr_with_target.plot(kind='barh', ax=ax, color=colors, edgecolor='black', alpha=0.8)
ax.axvline(x=0.03, color='red', linestyle='--', lw=1.5, label='Threshold = 0.03')
ax.set_title('|Correlation| with Cardio — Feature Selection', fontsize=14, fontweight='bold')
ax.set_xlabel('Absolute Correlation')
ax.legend()

for i, (val, name) in enumerate(zip(corr_with_target.values, corr_with_target.index)):
    ax.text(val + 0.005, i, f'{val:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

# แสดงผลการคัดเลือก
threshold = 0.03
selected = corr_with_target[corr_with_target >= threshold].index.tolist()
dropped = corr_with_target[corr_with_target < threshold].index.tolist()

print(f"Threshold: |correlation| >= {threshold}")
print(f"\n✓ SELECTED ({len(selected)} features): {selected}")
print(f"✗ DROPPED  ({len(dropped)} features): {dropped}")
print(f"\nเหตุผลที่ตัดออก:")
for feat in dropped:
    print(f"  - {feat}: corr = {corr_with_target[feat]:.4f} (ต่ำเกินไป ไม่ช่วยทำนาย)")
print(f"\n=> smoke, alco, gender แทบไม่มีความสัมพันธ์กับโรคหัวใจใน dataset นี้")
print(f"   สอดคล้องกับงานวิจัยที่ว่า direct measurement (ความดัน, cholesterol)")
print(f"   ทำนายได้ดีกว่า indirect risk factors")

In [ ]:
# กำหนด feature types — ใช้เฉพาะ features ที่ผ่านการคัดเลือก
# ตัดออก: gender (0.007), alco (0.009), smoke (0.016)
numerical_features = ['age', 'height', 'weight', 'ap_hi', 'ap_lo', 'bmi']
categorical_features = ['cholesterol', 'gluc', 'active']  # ตัด gender, smoke, alco

print(f"Selected features ({len(numerical_features) + len(categorical_features)}):")
print(f"  Numerical  ({len(numerical_features)}): {numerical_features}")
print(f"  Categorical ({len(categorical_features)}): {categorical_features}")
print(f"  Dropped: gender, smoke, alco (correlation < 0.03)")

X = df_clean[numerical_features + categorical_features]
y = df_clean['cardio']

# Train/Test split (80/20) พร้อม stratify เพื่อรักษา class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape[0]:,} records ({X_train.shape[1]} features)")
print(f"Test set:     {X_test.shape[0]:,} records ({X_test.shape[1]} features)")
print(f"\nClass balance in train: {y_train.value_counts(normalize=True).round(4).to_dict()}")
print(f"Class balance in test:  {y_test.value_counts(normalize=True).round(4).to_dict()}")
print("=> Stratify ทำให้ class balance เหมือนกันทั้ง train และ test")

In [ ]:
# สร้าง Preprocessing Pipeline
# - Numerical: StandardScaler (ทำให้ mean=0, std=1)
# - Categorical: passthrough (เป็น ordinal/binary อยู่แล้ว ไม่ต้อง encode)
#
# เหตุผลที่ใช้ StandardScaler:
# - Logistic Regression และ SVM ต้องการ features ที่ scale เดียวกัน
# - ap_hi, ap_lo มี range ต่างจาก age, bmi มาก
# - Tree-based models ไม่ได้รับผลจาก scaling แต่ก็ไม่เสียหาย

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', 'passthrough', categorical_features)
    ]
)

print("Preprocessor Pipeline:")
print(f"  Numerical features ({len(numerical_features)}): {numerical_features}")
print(f"    → StandardScaler")
print(f"  Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"    → Passthrough (ไม่ต้อง encode เพราะเป็น ordinal/binary)")

## 5. Baseline Model — Logistic Regression

เริ่มจาก Logistic Regression เป็น **baseline** เพราะ:
- เป็น model ที่เรียบง่ายและตีความได้ง่าย
- เหมาะกับ binary classification
- ใช้เป็นจุดอ้างอิงว่า model ที่ซับซ้อนกว่าดีขึ้นจริงหรือไม่

In [ ]:
from sklearn.linear_model import LogisticRegression

# สร้าง full pipeline: preprocessing + model
baseline_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Train
baseline_pipeline.fit(X_train, y_train)

# Predict
y_pred_base = baseline_pipeline.predict(X_test)
y_prob_base = baseline_pipeline.predict_proba(X_test)[:, 1]

# Evaluate
print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"\nAccuracy:  {accuracy_score(y_test, y_pred_base):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_base):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_base):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_base):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_base):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_base, target_names=['No CVD', 'Has CVD']))

In [ ]:
# Confusion Matrix + ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_base)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No CVD', 'Has CVD'], yticklabels=['No CVD', 'Has CVD'])
axes[0].set_title('Confusion Matrix — Logistic Regression', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_base)
auc_score = roc_auc_score(y_test, y_prob_base)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'Logistic Regression (AUC = {auc_score:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.500)')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=11)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.02])

plt.tight_layout()
plt.show()

# ตีความผลเชิง business
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (ไม่ป่วย ทำนายถูก):  {tn:,}")
print(f"True Positives (ป่วย ทำนายถูก):      {tp:,}")
print(f"False Positives (ไม่ป่วย แต่ทำนายว่าป่วย): {fp:,} — ส่งตรวจเพิ่มโดยไม่จำเป็น")
print(f"False Negatives (ป่วย แต่ทำนายว่าไม่ป่วย): {fn:,} — อันตราย! พลาดผู้ป่วยจริง")

In [ ]:
# Cross-validation เพื่อยืนยันว่า model stable ไม่ overfit
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(baseline_pipeline, X_train, y_train, cv=cv, scoring='accuracy')

print("5-Fold Cross-Validation (Logistic Regression):")
print(f"  Scores: {cv_scores.round(4)}")
print(f"  Mean:   {cv_scores.mean():.4f}")
print(f"  Std:    {cv_scores.std():.4f}")
print()
print(f"  Test Accuracy:  {accuracy_score(y_test, y_pred_base):.4f}")
print(f"  CV Mean:        {cv_scores.mean():.4f}")
print(f"  Difference:     {abs(accuracy_score(y_test, y_pred_base) - cv_scores.mean()):.4f}")
print("=> ค่าใกล้กัน แสดงว่า model ไม่ overfit")

## 6. Model Comparison

เปรียบเทียบหลาย algorithms เพื่อหา model ที่เหมาะสมที่สุด:
- **Logistic Regression** (baseline — linear model)
- **Random Forest** (ensemble — bagging)
- **Gradient Boosting** (ensemble — sequential boosting)
- **XGBoost** (ensemble — optimized boosting)

**เหตุผลที่เลือก 4 ตัวนี้:** ครอบคลุม approach ที่แตกต่างกัน ตั้งแต่ linear model ไปจนถึง ensemble methods ทั้ง bagging และ boosting เพื่อดูว่า model ประเภทไหนเหมาะกับ dataset นี้

> **หมายเหตุ:** ไม่ใช้ SVM เพราะ dataset มี 68K+ records ทำให้ SVM (O(n²) ถึง O(n³)) ใช้เวลานานเกินไป

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss', verbosity=0)
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"Training {name}...", end=" ")
    
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Cross-validation
    cv_acc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    cv_f1 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
    
    # Fit on full train, evaluate on test
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'pipeline': pipe,
        'cv_accuracy': cv_acc.mean(),
        'cv_accuracy_std': cv_acc.std(),
        'cv_f1': cv_f1.mean(),
        'cv_auc': cv_auc.mean(),
        'test_accuracy': accuracy_score(y_test, y_pred),
        'test_precision': precision_score(y_test, y_pred),
        'test_recall': recall_score(y_test, y_pred),
        'test_f1': f1_score(y_test, y_pred),
        'test_auc': roc_auc_score(y_test, y_prob),
        'y_pred': y_pred,
        'y_prob': y_prob
    }
    
    print(f"Done! CV Acc: {cv_acc.mean():.4f} | Test Acc: {accuracy_score(y_test, y_pred):.4f}")

print("\n✓ All models trained!")

In [ ]:
# สร้างตารางเปรียบเทียบ
results_df = pd.DataFrame({
    name: {
        'CV Accuracy': r['cv_accuracy'],
        'CV Accuracy Std': r['cv_accuracy_std'],
        'Test Accuracy': r['test_accuracy'],
        'Test Precision': r['test_precision'],
        'Test Recall': r['test_recall'],
        'Test F1': r['test_f1'],
        'Test ROC-AUC': r['test_auc']
    }
    for name, r in results.items()
}).T

print("Model Comparison Summary:")
print("=" * 90)
print(results_df.round(4).to_string())
print()

# หา best model
best_name = results_df['Test ROC-AUC'].idxmax()
print(f"✓ Best model by ROC-AUC: {best_name} ({results_df.loc[best_name, 'Test ROC-AUC']:.4f})")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
model_names = list(results.keys())

# 1. Accuracy comparison
metrics_to_plot = ['Test Accuracy', 'Test F1', 'Test ROC-AUC']
for idx, metric in enumerate(metrics_to_plot):
    values = [results_df.loc[name, metric] for name in model_names]
    bars = axes[idx].barh(model_names, values, color=colors, edgecolor='black', alpha=0.8)
    
    for bar, val in zip(bars, values):
        axes[idx].text(val + 0.002, bar.get_y() + bar.get_height()/2, 
                      f'{val:.4f}', va='center', fontweight='bold')
    
    axes[idx].set_title(metric, fontsize=14, fontweight='bold')
    axes[idx].set_xlim([0.65, 0.85])

plt.suptitle('Model Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves ของทุก models
fig, ax = plt.subplots(figsize=(8, 7))

for (name, r), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    ax.plot(fpr, tpr, color=color, lw=2, 
            label=f"{name} (AUC = {r['test_auc']:.3f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.500)')
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.legend(fontsize=11, loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

### วิเคราะห์ผลการเปรียบเทียบ

| Model | ข้อดี | ข้อจำกัด |
|-------|------|----------|
| **Gradient Boosting** | AUC สูงสุด (0.803), Accuracy สูงสุด | Train ช้ากว่า |
| XGBoost | ใกล้เคียง GB แต่เร็วกว่า | AUC ต่ำกว่า GB เล็กน้อย |
| Logistic Regression | เร็วที่สุด, ตีความง่าย | Performance ต่ำกว่า ensemble |
| Random Forest | Recall สูงสุด (พลาดผู้ป่วยน้อยสุด) | Accuracy & AUC ต่ำสุด |

**เลือก Gradient Boosting** เพราะให้ ROC-AUC สูงสุด ซึ่งสำคัญในบริบทการแพทย์เพราะวัดความสามารถในการแยกแยะผู้ป่วยจากผู้ไม่ป่วยได้ดีที่สุดในทุกระดับ threshold

## 7. Hyperparameter Tuning — Gradient Boosting

ใช้ **GridSearchCV** เพื่อหา hyperparameters ที่ดีที่สุดอย่างเป็นระบบ

**เหตุผลในการเลือก parameter ranges:**
- `n_estimators` [100, 200, 300]: จำนวน trees — มากขึ้นอาจดีขึ้นแต่ช้าลงและเสี่ยง overfit
- `max_depth` [3, 5, 7]: ความลึกของ tree — ลึกเกินจะ overfit, ตื้นเกินจะ underfit
- `learning_rate` [0.05, 0.1, 0.2]: อัตราการเรียนรู้ — ต่ำกว่า = เรียนรู้ช้าแต่ละเอียดกว่า
- `subsample` [0.8, 1.0]: สัดส่วนข้อมูลที่ใช้ train แต่ละ tree — < 1.0 ช่วยลด overfit

In [ ]:
from sklearn.model_selection import GridSearchCV

# Gradient Boosting Pipeline
gb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(random_state=42))
])

# Parameter grid
param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 5, 7],
    'classifier__learning_rate': [0.05, 0.1, 0.2],
    'classifier__subsample': [0.8, 1.0]
}

total_combinations = 3 * 3 * 3 * 2
print(f"Total combinations: {total_combinations}")
print(f"With 5-fold CV: {total_combinations * 5} fits")
print("Training...")

grid_search = GridSearchCV(
    gb_pipeline, param_grid, cv=5, scoring='roc_auc',
    n_jobs=-1, verbose=1, return_train_score=True
)

grid_search.fit(X_train, y_train)

print(f"\n✓ Best ROC-AUC (CV): {grid_search.best_score_:.4f}")
print(f"\nBest Parameters:")
for param, val in grid_search.best_params_.items():
    print(f"  {param.replace('classifier__', '')}: {val}")

In [ ]:
# Evaluate best model on test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

print("=" * 55)
print("BEST MODEL: Gradient Boosting (Tuned)")
print("=" * 55)
print(f"\nAccuracy:  {accuracy_score(y_test, y_pred_best):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_best):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_best):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_best):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_best):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_best, target_names=['No CVD', 'Has CVD']))

# เปรียบเทียบกับ baseline
print("=" * 55)
print("Improvement over Baseline (Logistic Regression):")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_best) - results['Logistic Regression']['test_accuracy']:+.4f}")
print(f"  F1-Score:  {f1_score(y_test, y_pred_best) - results['Logistic Regression']['test_f1']:+.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_prob_best) - results['Logistic Regression']['test_auc']:+.4f}")

In [ ]:
# Confusion Matrix + ROC Curve ของ best model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=axes[0],
            xticklabels=['No CVD', 'Has CVD'], yticklabels=['No CVD', 'Has CVD'])
axes[0].set_title('Confusion Matrix — Gradient Boosting (Tuned)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_best)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, 
             label=f'GB Tuned (AUC = {roc_auc_score(y_test, y_prob_best):.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
axes[1].set_title('ROC Curve — Best Model', fontsize=12, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

# ตีความเชิง business
tn, fp, fn, tp = cm.ravel()
total = tn + fp + fn + tp
print(f"\nการตีความเชิงปฏิบัติ:")
print(f"  จาก {total:,} คนที่ทดสอบ:")
print(f"  - ทำนายถูก {tn+tp:,} คน ({(tn+tp)/total*100:.1f}%)")
print(f"  - พลาดผู้ป่วยจริง (False Negative): {fn:,} คน → ต้องส่งตรวจเพิ่มเติม")
print(f"  - แจ้งเตือนเกิน (False Positive): {fp:,} คน → ส่งตรวจเพิ่มแต่ไม่เป็นอันตราย")

## 8. Feature Importance

ดูว่า features ตัวไหนมีผลต่อการทำนายมากที่สุดตามมุมมองของ Gradient Boosting

In [ ]:
# Feature importance จาก Gradient Boosting
all_features = numerical_features + categorical_features
gb_classifier = best_model.named_steps['classifier']
importances = gb_classifier.feature_importances_

# Sort by importance
feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=True)

plt.figure(figsize=(10, 7))
colors = ['#e74c3c' if v > feat_imp.median() else '#3498db' for v in feat_imp.values]
feat_imp.plot(kind='barh', color=colors, edgecolor='black', alpha=0.8)
plt.title('Feature Importance — Gradient Boosting', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')

for i, (val, name) in enumerate(zip(feat_imp.values, feat_imp.index)):
    plt.text(val + 0.005, i, f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print("Top 5 Features ที่สำคัญที่สุด:")
for i, (feat, val) in enumerate(feat_imp.sort_values(ascending=False).head(5).items(), 1):
    print(f"  {i}. {feat}: {val:.3f}")
print()
print("=> ap_hi (ความดันตัวบน) สำคัญที่สุด สอดคล้องกับความรู้ทางการแพทย์")
print("   ที่ว่าความดันโลหิตสูงเป็นปัจจัยเสี่ยงหลักของโรคหัวใจ")

## 9. Save Model

บันทึก model ที่ดีที่สุดเป็น `.pkl` เพื่อนำไป deploy บน Streamlit app

Pipeline ที่บันทึกจะรวมทั้ง preprocessing (StandardScaler) + model ไว้ด้วยกัน ทำให้สามารถรับ raw data แล้วทำนายได้เลยโดยไม่ต้อง preprocess แยก

In [ ]:
import joblib

# Save best model (full pipeline: preprocessor + classifier)
joblib.dump(best_model, 'cardio_model.pkl')

# Verify saved model works
loaded_model = joblib.load('cardio_model.pkl')
y_verify = loaded_model.predict(X_test)
assert (y_verify == y_pred_best).all(), "Loaded model gives different predictions!"

print("✓ Model saved successfully as 'cardio_model.pkl'")
print(f"  File size: {__import__('os').path.getsize('cardio_model.pkl') / 1024:.1f} KB")
print()
print("Pipeline components:")
for name, step in loaded_model.named_steps.items():
    print(f"  {name}: {type(step).__name__}")
print()
print(f"Expected input features ({len(numerical_features + categorical_features)}):")
print(f"  Numerical:    {numerical_features}")
print(f"  Categorical:  {categorical_features}")
print(f"  Dropped:      gender, smoke, alco (ไม่ต้องเก็บข้อมูลเหล่านี้)")